# MOVIE REDDIT SENTIMENT ENRICHMENT 

## Description
We will perform the same Reddit sentiment enrichment on the recommended movies from our within-domain and cross-domain recommender models. We will be using the same logic and algorithm as for book Reddit enrichments. This time we will be looking for posts and comments in three movie-themed subreddits ('movies', 'TrueFilm', 'moviesuggestions')

# Set Up

In [1]:
# Import the findspark module 
import findspark

# Initialize via the full spark path
findspark.init("/usr/local/spark/")

In [2]:
from pyspark.sql import SparkSession
import collections
from pyspark.sql.functions import col
from pyspark.sql.functions import from_json, col, explode, struct, collect_list
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.sql import Row

spark = SparkSession.builder \
   .master("local[8]") \
   .appName("Movie-Reddit-Enrichment") \
   .getOrCreate()

sc = spark.sparkContext

---
## Reddit Set Up
Load the necessary packages and connect to Reddit API with our credentials. 

!! Credentials (accessAPI and credentials) have to be created as separate python objects and imported !!

In [3]:
import sys
!{sys.executable} -m pip install praw --no-cache-dir
!{sys.executable} -m pip install -U textblob --no-cache-dir

Requirement already up-to-date: textblob in /opt/conda/lib/python3.8/site-packages (0.18.0.post0)


In [4]:
# Import reddit api wrapper and other neccessary packages:
import praw
from textblob import TextBlob
import json
from datetime import datetime
from kafka import KafkaProducer
import re
import time

# Import passwords and tokens:
import credentials
import accessAPI

# connect to api with credentials received from the website and user acc 
def reddit_connection():
    try: 
        reddit = praw.Reddit(client_id= accessAPI.CLIENT_ID,
                     client_secret=accessAPI.SECRET, 
                     password=credentials.password, 
                     username=credentials.username,
                     user_agent="MovieRecommendationSystem")
        return reddit

    except Exception as e:
        print("Error connecting to Reddit API")
        return None

---
## Reddit Data Collection
### Define helper functions that will be used later

In [5]:
def is_movie_mentioned(text, movie_title):
    # Checks if a specific movie is mentioned in text
    if not text or not movie_title:
        return False
    
    text_lower = text.lower()
    title_lower = movie_title.lower()
    
    # Direct match
    if title_lower in text_lower:
        return True
    
    # Match without punctuation
    clean_title = re.sub(r'[^\w\s]', '', title_lower)
    clean_text = re.sub(r'[^\w\s]', '', text_lower)
    
    return clean_title in clean_text

def find_mentioned_movies(text, movie_titles):
    # Find which movies from a list are mentioned in text
    mentioned = []
    for movie in movie_titles:
        if is_movie_mentioned(text, movie):
            mentioned.append(movie)
    return mentioned

def postData(post, movie_title, source_type):
    # Extract data from a Reddit post
    return {
        'type': 'post',
        'source': source_type,
        'title': post.title,
        'text': f"{post.title} {getattr(post, 'selftext', '')}",
        'score': post.score,
        'num_comments': post.num_comments,
        'created_utc': post.created_utc,
        'url': f"https://reddit.com{post.permalink}",
        'mentioned_movies': [movie_title],
        'subreddit': post.subreddit.display_name
    }

def commentData(comment, original_post, mentioned_movies):
    # Extract data from a Reddit comment
    return {
        'type': 'comment',
        'source': 'comment',
        'title': f"Comment on: {original_post['title']}",
        'text': comment.body,
        'score': comment.score,
        'num_comments': 0,
        'created_utc': getattr(comment, 'created_utc', 0),
        'url': f"https://reddit.com{comment.permalink}",
        'mentioned_movies': mentioned_movies,
        'subreddit': original_post['subreddit']
    }

def process_posts_movies(posts, movie_titles, mentions, source_type):
    # Process a list of posts to find the movie
    for post in posts:
        try:
            full_text = f"{post.title} {getattr(post, 'selftext', '')}"

            mentioned_movies = find_mentioned_movies(full_text, movie_titles)
            
            if mentioned_movies:
                for movie in mentioned_movies:
                    post_data = postData(post, movie, source_type)
                    
                    if movie not in mentions:
                        mentions[movie] = []

                    mentions[movie].append(post_data)
                    
        except Exception as e:
            continue

### Function to find Reddit posts mentioning movie titles
Look for posts mentioning the given movie title from the defined subreddits (in our case they will be 'movies', 'TrueFilm', 'moviesuggestions'). We search for the original title of the movie taken from the dataset, as well as different possible variations of the name, as sometimes the format of the title is  unlikely to be used on Reddit. E.g. if the title looks like "Mark of Cain, The", we'll be looking for both "Mark of Cain, The" and "The Mark of Cain", or if there are parentheses (usually with the name in the language of the original), we'll look for the full title, part before the parentheses and the one inside (e.g. "Heimat - A Chronicle of Germany (Heimat - Eine deutsche Chronik)", "Heimat - A Chronicle of Germany", "Heimat - Eine deutsche Chronik"). 

In [6]:
def movie_post(reddit, movie_title, subreddits):
    # Directly search for each movie title in the reddit posts

    all_posts = []
    seen_post_ids = set()  # Track unique post IDs to avoid duplicates
    MAX_TOTAL_POSTS = 150  # Stop once this limit is hit

    # Clean the titles of the movies and create variations of the title to look for:
    search_titles = [movie_title]  # Always include original title
    
    # Handle parentheses - get content before and inside parentheses
    parenth_match = re.search(r'^(.*?)\s*\(([^)]+)\)$', movie_title)
    if parenth_match:
        before_parenth = parenth_match.group(1).strip()
        inside_parenth = parenth_match.group(2).strip()
        
        # Add the part before parentheses
        if before_parenth:
            search_titles.append(before_parenth)
            
            # If the part before parentheses ends with ", The", also create "The" version (e.g. 'Tiger Brigades, The (Les brigades du Tigre)')
            if before_parenth.endswith(', The'):
                title_without_the = before_parenth[:-5]
                title_with_the_front = f"The {title_without_the}"
                search_titles.append(title_with_the_front)
        
        # Add the content inside parentheses
        if inside_parenth:
            search_titles.append(inside_parenth)
    else:
        # Handle ", The" at the end for titles without parentheses
        if movie_title.endswith(', The'):
            title_without_the = movie_title[:-5]  # Remove ", The"
            title_with_the_front = f"The {title_without_the}"
            search_titles.append(title_with_the_front)
    
    # Remove any duplicates
    search_titles = list(set([term.strip() for term in search_titles if term.strip()]))

    for name in subreddits:
        if len(all_posts) >= MAX_TOTAL_POSTS:
            print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
            break
        try:
            subreddit = reddit.subreddit(name)
            
            for search_title in search_titles:
                if len(all_posts) >= MAX_TOTAL_POSTS:
                    print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
                    break
                try:
                    posts_for_title = 0 # counter of the number of posts collected for a title 
                    
                    # Looks for posts with this movie title
                    results = subreddit.search(
                        search_title, 
                        limit=100,
                        sort='relevance'
                    )
                    for post in results:
                        if len(all_posts) >= MAX_TOTAL_POSTS:
                            print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
                            break

                        # Check if we've already seen this post ID
                        if post.id in seen_post_ids:
                            continue  # Skip duplicate post
                        
                        # Add post ID to seen set
                        seen_post_ids.add(post.id)

                        # using getattr ensures that if a post for some reason doesn't have selftext, no errors arise: 
                        text = f"{post.title} {getattr(post, 'selftext', '')}"  
                        post_data = postData(post, movie_title, 'search')
                        all_posts.append(post_data)
                        posts_for_title += 1
                    print(f"Found {posts_for_title} posts for '{search_title}' in r/{name}")
                    
                    time.sleep(1)

                except Exception as e:
                    print(f"Search failed for '{search_title}': {e}")
                    continue

        except Exception as e:
            print(f"Cannot access r/{name}: {e}")
            continue

    print(f"Total posts found in the search: {len(all_posts)}")
    return all_posts

### Function to find Reddit comments mentioning movie titles
Look for comments to the posts collected previously and collect the ones mentioning our movie. Once again we use use the same variations of the movie title to broaden our search and, hopefully, collect more data. 

In [7]:
# Collect comments from the relevant posts:
def movie_comments(reddit, posts_data, movie_title):
    
    # Generate the same title variations as in movie_post():
    search_titles = [movie_title]  # Always include original title
    
    # Handle parentheses - get content before and inside parentheses
    parenth_match = re.search(r'^(.*?)\s*\(([^)]+)\)$', movie_title)
    if parenth_match:
        before_parenth = parenth_match.group(1).strip()
        inside_parenth = parenth_match.group(2).strip()
        
        # Add the part before parentheses
        if before_parenth:
            search_titles.append(before_parenth)
            
            # If the part before parentheses ends with ", The", also create "The" version (e.g. 'Tiger Brigades, The (Les brigades du Tigre)')
            if before_parenth.endswith(', The'):
                title_without_the = before_parenth[:-5]
                title_with_the_front = f"The {title_without_the}"
                search_titles.append(title_with_the_front)
        
        # Add the content inside parentheses
        if inside_parenth:
            search_titles.append(inside_parenth)
    else:
        # Handle ", The" at the end for titles without parentheses
        if movie_title.endswith(', The'):
            title_without_the = movie_title[:-5]  # Remove ", The"
            title_with_the_front = f"The {title_without_the}"
            search_titles.append(title_with_the_front)
    
    # Remove any duplicates or empty terms
    search_titles = list(set([term.strip() for term in search_titles if term.strip()]))
    
    # Add limits of the number of posts for which comments are collected, to avoid long processing and overloading it:
    if len(posts_data) > 40: # if there are more than 40 posts, use only the top 15 to collect comments:
        max_posts = 15
    else:
        max_posts = len(posts_data) # otherwise process all of them

    # Sort by scores to take the most popular comments (they would most probably have more comments):
    sorted_posts = sorted(posts_data, key=lambda x: x['score'], reverse=True)
    posts_to_process = sorted_posts[:max_posts]
    
    all_comments = []
    seen_comment_ids = set()  # Track unique comment IDs to avoid duplicates
    
    for post_data in posts_to_process:
        if len(all_comments) >= 50: # stop if we already collected at least 50 comments from other posts
            break 
        try:
            # Get the url of the post and load the comments:
            post = reddit.submission(url=post_data['url'])
            post.comments.replace_more(limit=3) # limit to 3 levels of threads in the comments
 
            comment_count_per_post = 0
            for comment in post.comments.list():
                if len(all_comments) >= 50: # stop if 50 comments reached
                    break 
                try:
                    # Check if the comment has a body and is longer than 10, so that it's meaningful:
                    if hasattr(comment, 'body') and len(comment.body) > 10:
                        # Check if we've already seen this comment ID
                        if comment.id in seen_comment_ids:
                            continue  # Skip duplicate comment
                        
                        # Check if it mentions any of the movie title variations:
                        mentioned_movies = find_mentioned_movies(comment.body, search_titles)
                        if mentioned_movies:
                            # Add comment ID to seen set
                            seen_comment_ids.add(comment.id)
                            
                            comment_data = commentData(comment, post_data, [movie_title])  # Store original title
                            all_comments.append(comment_data)
                            comment_count_per_post += 1
                        if comment_count_per_post >= 50:
                            break
                except Exception as e:
                    continue

            time.sleep(1)

        except Exception as e:
            print(f"Failed to process comments: {e}")
            continue

    print(f"The number of comments collected for '{movie_title}' is {len(all_comments)}")
    return all_comments

### Comprehensive Movie Reddit Collection Function
Look for both posts and comments mentioning a given movie title in the chosen subreddits. We are limiting our search to 150 posts and 50 comments maximum, to try and get enough meaningful data and to not exceed Reddit API limitations, as well as to decrease the time needed for making the final recommendations. 

In [8]:
# Function to collect Reddit posts and comments for multiple movie titles
# Returns dictionary with movie titles as keys and collected data as values
import builtins
def collect_reddit_movie_data(reddit, movie_titles, subreddits=None):
    # Use these subreddits, if others aren't given:
    if subreddits is None:
        subreddits = ['movies', 'TrueFilm', 'moviesuggestions']
    
    # Initialize results dictionary:
    results = {}
    
    for i, movie_title in enumerate(movie_titles, 1):
        print(f"\n--- Processing movie {i}/{len(movie_titles)}: '{movie_title}' ---")
        
        # Step 1: Find posts
        all_posts = movie_post(reddit, movie_title, subreddits)
        
        # Step 2: Get comments from found posts
        all_comments = [] 
        if all_posts:
            all_comments = movie_comments(reddit, all_posts, movie_title)
        
        # Print collection results
        n_items = len(all_posts) + len(all_comments)
        print(f"Collection complete: {len(all_posts)} posts, {len(all_comments)} comments (total: {n_items})")
        
        # Store results for this movie:
        results[movie_title] = {
            'posts': all_posts,
            'comments': all_comments,
            'total_posts': len(all_posts),
            'total_comments': len(all_comments),
            'total_items': len(all_posts) + len(all_comments)
        }
    
    # Print summary
    print(f"\n=== COLLECTION SUMMARY ===")
    total_posts_all = builtins.sum([data['total_posts'] for data in results.values()])
    total_comments_all = builtins.sum([data['total_comments'] for data in results.values()])
    print(f"Total across all movies: {total_posts_all} posts, {total_comments_all} comments")

    for movie_title, data in results.items():
        print(f"'{movie_title}': {data['total_posts']} posts, {data['total_comments']} comments")

    return results

---
# Sentiment Analysis
We will be using the TextBlob library for sentiment analysis and the parameter we'll focus on is polarity. It doesn't make sense to use subjectivity as what we're looking for are the subjective opinions of people. Polarity is exactly what we need in this case, as it captures the sentiment direction and strength, ranging from -1 (negative) to +1 (positive).

In [9]:
# Load additional necessary packages
import os
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

### Define helper functions

In [10]:
# Define preprocessing helper function for the text:
def preprocess_text_spark(df, text_column):
    # Remove empty or null values:
    df = df.filter(col(text_column).isNotNull())
    df = df.filter(col(text_column) != "")
    
    # Remove URLs, mentions, hashtags, special characters and os on
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'http\S+', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'@\w+', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, '#', '')) 
    df = df.withColumn(text_column, F.regexp_replace(text_column, ':', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\n', ' '))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\*', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\s+', ' '))
    df = df.withColumn(text_column, F.trim(text_column))
    df = df.withColumn(text_column, F.lower(text_column))
    
    return df

# Define a polarity detection function using TextBlob:
def apply_sentiment_analysis(df, text_column):
    # Define a helper polarity detection function that returns the polarity score (from -1 to 1) using TextBlob:
    def polarity_detection(text):
        if text is None or text.strip() == "":
            return 0.0
        try:
            return float(TextBlob(text).sentiment.polarity)
        except:
            return 0.0
    
    # Create UDF for polarity detection (so that it can be applied to a PySpark dataframe with the data from Reddit)
    polarity_udf = udf(polarity_detection, FloatType())

    # Apply sentiment analysis to a PySpark dataframe with a text column:
    df = df.withColumn("polarity", polarity_udf(col(text_column)))
    return df

### Function to convert reddit data to Spark
We create a function that converts the Reddit data previously collected in the form of a nested dictionary structure containing posts and comments organized by movie titles into PySpark DataFrames. It extracts and cleans text fields, filters out empty content, and standardizes the fields into rows with movie_title, text, type, and score. This function helps prepare the data for sentiment analysis and using Spark makes it scalable, potentially enabling large amounts of data.

In [11]:
# Convert reddit data to Spark dataframes:
def process_reddit_data_to_spark(spark, sentiment_data):
    movie_dataframes = {}
    
    for movie_title, data in sentiment_data.items():
        # Combine posts and comments into a single list
        all_texts = []
        
        # Use field names from the structure defined in postData function:
        for post in data['posts']:
            if 'text' in post and post['text'] and post['text'].strip():
                all_texts.append(Row(
                    movie_title=movie_title,
                    text=post['text'].strip(),
                    type='post',
                    score=post.get('score', 0)
                ))
        
        # Use field names from the structure defined in commentData function:
        for comment in data['comments']:
            if 'text' in comment and comment['text'] and comment['text'] != '[removed]':
                all_texts.append(Row(
                    movie_title=movie_title,
                    text=comment['text'].strip(),
                    type='comment',
                    score=comment.get('score', 0)
                ))
        
        # Create Spark DataFrame:
        if all_texts:
            try:
                movie_df = spark.createDataFrame(all_texts)
                movie_dataframes[movie_title] = movie_df
            except Exception as e:
                print(f"Error creating DataFrame for {movie_title}: {e}")
                movie_dataframes[movie_title] = None
        else:
            print(f"No valid text found for {movie_title}")
            movie_dataframes[movie_title] = None
            
    return movie_dataframes

In [12]:
# Function that will be used to take a single movie's DataFrame and run the sentiment analysis on it:
def analyze_movie_sentiment(spark, movie_df):
    # If the dataframe doesn't exist, returns None:
    if movie_df is None:
        return None
    
    # Preprocess text: clean it using the preprocess_text_spark defined earlier (removing URLs, mentions, hashtags, special characters, and so on)
    processed_df = preprocess_text_spark(movie_df, 'text')
    
    # Apply sentiment analysis: use our UDF to add a 'polarity' column with sentiment scores (-1 to +1):
    sentiment_df = apply_sentiment_analysis(processed_df, 'text')
    
    # Calculate the metrics needed:
    metrics = sentiment_df.agg(
        F.avg('polarity').alias('avg_polarity'), # calculate the average polarity for a movie
        F.count('polarity').alias('total_texts') # find the total number of post+comments analyzed
    ).collect()[0]
    
    # Returns overall sentiment and the number posts/comments analyzed for a movie :
    return metrics 

## Combining model recommendations with sentiment analysis results

We're using the same logic as for books.

**Plan:**
- Step 1: Normalize polarity from [-1,1] to [0,1] 
- Step 2: Calculate volume multiplier based on post/comment counts 
- Step 3: Apply volume multiplier to normalized polarity 
- Step 4: Normalize Reddit scores to model score range, then combine using 0.6/0.4 weights
- Step 5: Penalize movies with no Reddit data (because that means they're not popular)

**calculate_reddit_scores():** Get sentiment data for each movie, normalizes polarity to [0,1] using: (polarity + 1) / 2, calculate volume multiplier based on text count (more texts = more reliable), return final Reddit score (normalized_polarity * volume_multiplier).

**calculate_volume_multiplier():** Assigns multipliers based on volume levels, 150+ texts = 1.0 (maximum reliability), down to 5 texts = 0.3 (minimum reliability). 

**combine_model_and_reddit_scores():** First normalize Reddit scores to the model score range, then combine using 0.6/0.4 (Models/Reddit) weights, apply the 0.05 penalty when no Reddit data exists, ensure scores don't go negative.


### Calculating the sentiment analysis coefficients

In [13]:
# Calculate volume multiplier based on number of posts/comments.
# Higher volume means higher multiplier because the analysis is more reliable.
def calculate_volume_multiplier(total_texts):
    if total_texts >= 150:  # Very high (close to max 200)
        return 1.0
    elif total_texts >= 100: 
        return 0.9
    elif total_texts >= 50:   
        return 0.8
    elif total_texts >= 20:   
        return 0.7
    elif total_texts >= 10:   
        return 0.6
    elif total_texts >= 5:    # Very low volume
        return 0.5
    else:  # Extremely low volume
        return 0.3

    
# Calculate Reddit sentiment scores for each movie.
# Return scores from 0 to 1 where 1 is maximum.
def calculate_reddit_scores(sentiment_data, spark):
    # Convert to Spark DataFrames:
    movie_dataframes = process_reddit_data_to_spark(spark, sentiment_data)
    
    reddit_scores = {}
    
    for movie_title, movie_df in movie_dataframes.items():
        print()
        print(f"Processing {movie_title}...")
        
        # Get sentiment metrics
        # If no data:
        if movie_df is None:
            reddit_scores[movie_title] = {
                'score': 0.0,  
                'total_texts': 0,
                'avg_polarity': 0.0,
                'volume_multiplier': 0.0
            }
            continue
            
        metrics = analyze_movie_sentiment(spark, movie_df)
        
        if metrics is None or metrics['total_texts'] == 0:
            reddit_scores[movie_title] = {
                'score': 0.0,  
                'total_texts': 0,
                'avg_polarity': 0.0,
                'volume_multiplier': 0.0
            }
            continue
        
        # Extract values
        avg_polarity = float(metrics['avg_polarity'])
        total_texts = int(metrics['total_texts'])
        
        # Step 1: normalize polarity from [-1,1] to [0,1]:
        normalized_polarity = (avg_polarity + 1) / 2
        
        # Step 2: calculate volume multiplier based on popularity
        # Max possible: 200 texts (150 posts + 50 comments)
        volume_multiplier = calculate_volume_multiplier(total_texts)
        
        # Step 3: apply volume multiplier
        reddit_score = normalized_polarity * volume_multiplier
        
        reddit_scores[movie_title] = {
            'score': reddit_score,
            'total_texts': total_texts,
            'avg_polarity': avg_polarity,
            'volume_multiplier': volume_multiplier
        }
        
        print(f"  Polarity: {avg_polarity:.3f} -> Normalized: {normalized_polarity:.3f}")
        print(f"  Volume: {total_texts} texts -> Multiplier: {volume_multiplier:.3f}")
        print(f"  Final Reddit score: {reddit_score:.3f}")
    
    return reddit_scores

### Combining the scores

In [14]:
# Combine model scores with Reddit sentiment scores
# no_data_penalty is the penalty that is subtracted when there's no data (to reflect that this movie is not popular)
# We consider our models to be more reliable, so their weight it higher (0.6)
def combine_model_and_reddit_scores(model_recommendations, reddit_scores, 
                                  model_weight=0.6, reddit_weight=0.4, 
                                  no_data_penalty=0.05):
    combined_scores = []
    
    # Collect all Reddit scores to normalize them
    reddit_values = []
    model_values = []
    
    for movie_title, model_score in model_recommendations:
        reddit_data = reddit_scores.get(movie_title, {})
        reddit_score = reddit_data.get('score', 0.0)
        total_texts = reddit_data.get('total_texts', 0)
        
        if total_texts > 0:  # Only include movies with Reddit data for normalization
            reddit_values.append(reddit_score)
            model_values.append(model_score) 
    
    
    # Calculate ranges
    if reddit_values:
        min_reddit = builtins.min(reddit_values)
        max_reddit = builtins.max(reddit_values)
        min_model = builtins.min(model_values)
        max_model = builtins.max(model_values)
        
        reddit_range = max_reddit - min_reddit
        model_range = max_model - min_model
        
        print(f"Reddit scores range: {min_reddit:.3f} to {max_reddit:.3f}")
        print(f"Model scores range: {min_model:.3f} to {max_model:.3f}")
    else:
        # No Reddit data available
        min_reddit = max_reddit = reddit_range = 0
        min_model = max_model = model_range = 0
        
        
    # Combine scores with normalized Reddit values:
    for movie_title, model_score in model_recommendations:
        reddit_data = reddit_scores.get(movie_title, {})
        reddit_score = reddit_data.get('score', 0.0)
        total_texts = reddit_data.get('total_texts', 0)
        
        if total_texts == 0:
            # No Reddit data: apply penalty to model score
            final_score = model_score - no_data_penalty
            print(f"{movie_title}: No Reddit data, applying penalty of {no_data_penalty}")
        else:
            # Normalize Reddit score to model score range
            if reddit_range > 0:
                # Scale Reddit score to model range
                normalized_reddit = min_model + (reddit_score - min_reddit) / reddit_range * model_range
            else:
                # If no variation in Reddit scores, use middle of model range
                normalized_reddit = (min_model + max_model) / 2
                
            # Combine normalized scores
            final_score = (model_weight * model_score) + (reddit_weight * normalized_reddit)
            print(f"{movie_title}: Model({model_score:.3f}) + Reddit({reddit_score:.3f} -> {normalized_reddit:.3f}) = {final_score:.3f}")
        
        # Ensure score doesn't go below 0
        final_score = builtins.max(0.0, final_score)

        combined_scores.append([movie_title, final_score])
    
    # Sort by final score:
    combined_scores.sort(key=lambda x: x[1], reverse=True)
    
    return combined_scores

--- 
# Within-Domain
## Load recommendations from our models
The results were previously saved to a csv file (movie_user_recs.csv).

In [15]:
# Load the CSV file
try: 
    recs_raw = spark.read.csv("../Models/models_results/movie_user_recs.csv", header=True, inferSchema=True)
except Exception as e:
    print(f"Error loading file: {e}")
    raise
# Show a few rows to confirm
recs_raw.show(3)

+-------+---------------------------+
|user_id|hybrid_recommendations_json|
+-------+---------------------------+
|   7253|       [[86892.0,0.66435...|
|  26755|       [[78499.0,0.74189...|
|  28088|       [[78499.0,0.65407...|
+-------+---------------------------+
only showing top 3 rows



In [16]:
# Loading movie metadata:
try: 
    movie_metadata = spark.read.csv("../Models/within_domain_data/movies_metadata_filtered_1980_plus.csv", header = True)
except Exception as e:
    print(f"Error loading file: {e}")
    raise

movie_metadata.show(3)

+---------+-------------+-------+-------+--------------------+----------------+----+
|avgRating|   directedBy| imdbId|item_id|            starring|           title|year|
+---------+-------------+-------+-------+--------------------+----------------+----+
|  3.89146|John Lasseter|0114709|      1|Tim Allen, Tom Ha...|       Toy Story|1995|
|  3.26605| Joe Johnston|0113497|      2|Jonathan Hyde, Br...|         Jumanji|1995|
|  3.17146|Howard Deutch|0113228|      3|Jack Lemmon, Walt...|Grumpier Old Men|1995|
+---------+-------------+-------+-------+--------------------+----------------+----+
only showing top 3 rows



In [17]:
# Define schema for JSON-like array field
recommendation_schema = ArrayType(ArrayType(DoubleType()))

# Parse JSON string column into nested array
recs_parsed = recs_raw.withColumn(
    "hybrid_recommendations",
    from_json(col("hybrid_recommendations_json"), recommendation_schema)
).drop("hybrid_recommendations_json")

# Filter out rows where parsing failed
recs_parsed = recs_parsed.filter(col("hybrid_recommendations").isNotNull())

# Explode recommendations: one row per recommended item
recs_exploded = recs_parsed.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).cast("long").alias("item_id"),  # cast ID to long
    col("rec").getItem(1).alias("score")
)

# Join with movie metadata to get titles
recs_with_titles = recs_exploded.join(
    movie_metadata,
    on="item_id",
    how="left"
)

# Replace item_id with struct(title, score) instead of array
recs_with_titles = recs_with_titles.withColumn(
    "rec_with_title",
    struct("title", "score")
)

# Group back by user_id
final_user_recs_with_titles = recs_with_titles.groupBy("user_id").agg(
    collect_list("rec_with_title").alias("hybrid_recommendations_with_titles")
)

---
# Applying sentiment analysis to users
We'll be  demonstrating the Reddit sentiment enrichment on recommendations for 3 users, but with enough time and without the API limitations, the algorithm can be scaled up, and applied to any amount of users and recommendation.

In [18]:
# Show for 3 users with 15 recommended movies each. The same can be done for the other users as well.
users = final_user_recs_with_titles.limit(3)  
users_data = users.collect()

for row in users_data:
    print(f"\nUser ID: {row['user_id']}")
    print("Top 15 Recommendation:")
    for title_score in row['hybrid_recommendations_with_titles'][:15]:
        print(f"  - {title_score['title']}: {title_score['score']:.4f}")

print()
print(f"=== REDDIT SENTIMENT ANALYSIS ===")
print(f"Processing {len(users_data)} user for demonstration...")

# Initialize Reddit connection (using your existing function)
reddit = reddit_connection()
if reddit is None:
    print("Failed to connect to Reddit API")
    exit()


User ID: 7253
Top 15 Recommendation:
  - The Man from Nowhere: 0.6644
  - Children of Heaven, The (Bacheha-Ye Aseman): 0.6644
  - Toy Story 3: 0.6644
  - Stay: 0.6553
  - Death on the Staircase (Soupçons): 0.6317
  - The Jinx: The Life and Deaths of Robert Durst: 0.6317
  - The Wrecking Crew: 0.6317
  - Blackfish: 0.6317
  - O.J.: Made in America: 0.6317
  - Mother!: 0.6317
  - We Need to Talk About Kevin: 0.5511
  - Blue Jasmine: 0.5511
  - Bourne Identity, The: 0.5463
  - Bandits: 0.5408
  - Medicine Man: 0.5350

User ID: 26755
Top 15 Recommendation:
  - Toy Story 3: 0.7419
  - The Man from Nowhere: 0.7419
  - Children of Heaven, The (Bacheha-Ye Aseman): 0.7419
  - Stay: 0.6944
  - Death on the Staircase (Soupçons): 0.6591
  - O.J.: Made in America: 0.6591
  - The Wrecking Crew: 0.6591
  - The Jinx: The Life and Deaths of Robert Durst: 0.6591
  - Blackfish: 0.6591
  - Mother!: 0.6591
  - Haywire: 0.6476
  - Bourne Identity, The: 0.6357
  - Blue Jasmine: 0.6016
  - We Need to Talk Ab

### Collect Reddit Data

In [19]:
all_movie_titles = []
for user_row in users_data:
    for rec in user_row['hybrid_recommendations_with_titles']:
        if rec['title'] is not None and rec['title'] not in all_movie_titles:
            all_movie_titles.append(rec['title'])

# Collect Reddit data once for all movie
reddit_data = collect_reddit_movie_data(
    reddit,
    list(all_movie_titles)
)


--- Processing movie 1/18: 'The Man from Nowhere' ---
Found 100 posts for 'The Man from Nowhere' in r/movies
Reached maximum limit of 150 posts. Stopping search.
Found 50 posts for 'The Man from Nowhere' in r/TrueFilm
Reached maximum limit of 150 posts. Stopping search.
Total posts found in the search: 150
The number of comments collected for 'The Man from Nowhere' is 25
Collection complete: 150 posts, 25 comments (total: 175)

--- Processing movie 2/18: 'Children of Heaven, The (Bacheha-Ye Aseman)' ---
Found 100 posts for 'The Children of Heaven' in r/movies
Found 0 posts for 'Children of Heaven, The' in r/movies
Found 0 posts for 'Bacheha-Ye Aseman' in r/movies
Found 0 posts for 'Children of Heaven, The (Bacheha-Ye Aseman)' in r/movies
Found 35 posts for 'The Children of Heaven' in r/TrueFilm
Found 0 posts for 'Children of Heaven, The' in r/TrueFilm
Found 0 posts for 'Bacheha-Ye Aseman' in r/TrueFilm
Found 0 posts for 'Children of Heaven, The (Bacheha-Ye Aseman)' in r/TrueFilm
Reach

### Example of data collected for the first movie for each user

In [20]:
# Preview some sample Reddit data for the first movie from each user
for user_row in users_data:
    user_id = user_row['user_id']
    
    # Get first movie with a title for this user
    first_movie_title = None
    for rec in user_row['hybrid_recommendations_with_titles']:
        if rec['title'] is not None:
            first_movie_title = rec['title']
            break
    
    if first_movie_title and first_movie_title in reddit_data:
        print(f"\n=== Raw Data for USER {user_id}: '{first_movie_title}' ===")
        movie_data = reddit_data[first_movie_title]
        print(f"\n-- Raw Posts --")
        for i, post in enumerate(movie_data['posts'][:1], 1):
            print(f"\nPost #{i}")
            print(f"Type       : {post['type']}")
            print(f"Source     : {post['source']}")
            print(f"Title      : {post['title']}")
            print(f"Score      : {post['score']}")
            print(f"Comments   : {post['num_comments']}")
            print(f"Created    : {post['created_utc']}")
            print(f"Subreddit  : {post['subreddit']}")
            print(f"URL        : {post['url']}")
            print(f"Mentioned  : {post['mentioned_movies']}")
            print(f"Text:\n{post['text'][:1500] + '...' if len(post['text']) > 1500 else post['text']}")
            print("-" * 80)
        print(f"\n-- Raw Comments --")
        for i, comment in enumerate(movie_data['comments'][:1], 1):
            print(f"\nComment #{i}")
            print(f"Type       : {comment['type']}")
            print(f"Source     : {comment['source']}")
            print(f"Title      : {comment['title']}")
            print(f"Score      : {comment['score']}")
            print(f"Comments   : {comment['num_comments']}")
            print(f"Created    : {comment['created_utc']}")
            print(f"Subreddit  : {comment['subreddit']}")
            print(f"URL        : {comment['url']}")
            print(f"Mentioned  : {comment['mentioned_movies']}")
            print(f"Text:\n{comment['text']}")
            print("-" * 80)


=== Raw Data for USER 7253: 'The Man from Nowhere' ===

-- Raw Posts --

Post #1
Type       : post
Source     : search
Title      : If you like John Wick then you should really watch the Korean film 'Man from Nowhere'.
Score      : 20076
Comments   : 1475
Created    : 1487469450.0
Subreddit  : movies
URL        : https://reddit.com/r/movies/comments/5uw0gc/if_you_like_john_wick_then_you_should_really/
Mentioned  : ['The Man from Nowhere']
Text:
If you like John Wick then you should really watch the Korean film 'Man from Nowhere'. The storylines aren't that far off and it has some of the best fighting scenes on film especially the final fight scene, [which you can view here but it is a slight spoiler if you watch it all.](https://www.youtube.com/watch?v=bSxGEUSxPHI)

It's one of my favourite movies and has some great character development, storyline and the fight scenes are incredible.

--------------------------------------------------------------------------------

-- Raw Comments --

### Compute Reddit scores and combine with Model scores

In [21]:
# Compute Reddit scores (for sentiment analysis) using the data collected in reddit_data:
reddit_scores_cache = calculate_reddit_scores(
    reddit_data,
    spark
)


Processing The Man from Nowhere...
  Polarity: 0.139 -> Normalized: 0.570
  Volume: 175 texts -> Multiplier: 1.000
  Final Reddit score: 0.570

Processing Children of Heaven, The (Bacheha-Ye Aseman)...
  Polarity: 0.121 -> Normalized: 0.560
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.560

Processing Toy Story 3...
  Polarity: 0.122 -> Normalized: 0.561
  Volume: 200 texts -> Multiplier: 1.000
  Final Reddit score: 0.561

Processing Stay...
  Polarity: 0.067 -> Normalized: 0.534
  Volume: 200 texts -> Multiplier: 1.000
  Final Reddit score: 0.534

Processing Death on the Staircase (Soupçons)...
  Polarity: 0.058 -> Normalized: 0.529
  Volume: 50 texts -> Multiplier: 0.800
  Final Reddit score: 0.423

Processing The Jinx: The Life and Deaths of Robert Durst...
  Polarity: 0.103 -> Normalized: 0.552
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.552

Processing The Wrecking Crew...
  Polarity: 0.146 -> Normalized: 0.573
  Volume: 55 texts -> Multipl

In [22]:
# Process the user using the computed reddit_scores_cache and our combine_model_and_reddit_scores function:
def process_user_with_sentiment(user_id, user_recs, reddit_scores_cache):
    print(f"\n--- Processing User {user_id} ---")
    
    # Extract movie titles from recommendations
    movie_titles = []
    model_recommendations = []
    
    for rec in user_recs:
        if rec['title'] is not None:
            movie_titles.append(rec['title'])
            model_recommendations.append((rec['title'], rec['score']))
    
    print(f"Found {len(movie_titles)} movies to analyze")
    print("Original 15 recommendations:")
    for i, (title, score) in enumerate(model_recommendations[:15]):
        print(f"  {i+1}. {title[:60]}... - Score: {score:.4f}")
    
    print("\nCombining model scores with Reddit sentiment analysis...")
    
    try:
        combined_scores = combine_model_and_reddit_scores(
            model_recommendations,
            reddit_scores_cache,
            model_weight=0.6,
            reddit_weight=0.4,
            no_data_penalty=0.05
        )
        return combined_scores
    except Exception as e:
        print(f"Error combining recommendations: {e}")
        return model_recommendations

# Final Recommendations - Within Domain

In [23]:
# Process users and print results using process_user_with_sentiment:
print("\n" + "="*80)
print("PROCESSING EXAMPLE USER")
print("="*80)
processed_users = []
for i, user_row in enumerate(users_data):
    user_id = user_row['user_id']
    user_recs = user_row['hybrid_recommendations_with_titles']
    
    try:
        # Use our function that reddit_scores_cache had already been collected
        sentiment_enhanced_recs = process_user_with_sentiment(
            user_id, 
            user_recs, 
            reddit_scores_cache 
        )
        
        # Store results:
        processed_users.append({
            'user_id': user_id,
            'original_recommendations': [(rec['title'], rec['score']) for rec in user_recs if rec['title'] is not None],
            'sentiment_enhanced_recommendations': sentiment_enhanced_recs
        })
        
        # Show comparison of previous scores with the final ones:
        print(f"\n{'='*60}")
        print(f"RESULTS FOR USER {user_id}")
        print(f"{'='*60}")
        
        print("\nTOP 15 COMPARISON:")
        print(f"{'Rank':<4} | {'Original Score':<13} | {'Enhanced Score':<13} | {'Change':<8} | {'Title'}")
        print("-" * 100)
        
        original_recs = processed_users[-1]['original_recommendations'][:15]
        enhanced_recs = sentiment_enhanced_recs[:15]
        
        for j in range(builtins.min(15, len(original_recs), len(enhanced_recs))):
            orig_score = original_recs[j][1]
            enh_score = enhanced_recs[j][1]
            change = enh_score - orig_score
            change_str = f"{change:+.4f}"
            title = enhanced_recs[j][0][:50] + "..." if len(enhanced_recs[j][0]) > 50 else enhanced_recs[j][0]
            
            print(f"{j+1:<4} | {orig_score:<13.4f} | {enh_score:<13.4f} | {change_str:<8} | {title}")
        
    except Exception as e:
        print(f"Error processing user {user_id}: {e}")
        continue
        

# Create PySpark DataFrame from results:

# Prepare data for PySpark DataFrame
spark_data = []
for user_data in processed_users:
    user_id = user_data['user_id']
    
    # Flatten original recommendations
    original_titles = [rec[0] for rec in user_data['original_recommendations']]
    original_scores = [rec[1] for rec in user_data['original_recommendations']]
    
    # Flatten enhanced recommendations
    enhanced_titles = [rec[0] for rec in user_data['sentiment_enhanced_recommendations']]
    enhanced_scores = [rec[1] for rec in user_data['sentiment_enhanced_recommendations']]
    
    spark_data.append({
        'user_id': user_id,
        'original_titles': original_titles,
        'original_scores': original_scores,
        'enhanced_titles': enhanced_titles,
        'enhanced_scores': enhanced_scores,
        'top_15_recommendations': enhanced_titles[:15] 
    })

# Define schema for better performance
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("original_titles", ArrayType(StringType()), True),
    StructField("original_scores", ArrayType(FloatType()), True),
    StructField("enhanced_titles", ArrayType(StringType()), True),
    StructField("enhanced_scores", ArrayType(FloatType()), True),
    StructField("top_15_recommendations", ArrayType(StringType()), True)
])

# Create PySpark DataFrame
recommendations_df = spark.createDataFrame(spark_data, schema)

print("\nPySpark DataFrame Preview:")
recommendations_df.show(truncate=False)

print()
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS ORDER")
print("="*80)

# Collect results
results = recommendations_df.collect()

for row in results:
    user_id = row['user_id']
    top_15 = row['top_15_recommendations']
    
    print(f"\nTOP 15 MOVIE RECOMMENDATIONS FOR USER {user_id}")
    print("-" * 50)
    for i, title in enumerate(top_15, 1):
        print(f"{i:2d}. {title}")


PROCESSING EXAMPLE USER

--- Processing User 7253 ---
Found 15 movies to analyze
Original 15 recommendations:
  1. The Man from Nowhere... - Score: 0.6644
  2. Children of Heaven, The (Bacheha-Ye Aseman)... - Score: 0.6644
  3. Toy Story 3... - Score: 0.6644
  4. Stay... - Score: 0.6553
  5. Death on the Staircase (Soupçons)... - Score: 0.6317
  6. The Jinx: The Life and Deaths of Robert Durst... - Score: 0.6317
  7. The Wrecking Crew... - Score: 0.6317
  8. Blackfish... - Score: 0.6317
  9. O.J.: Made in America... - Score: 0.6317
  10. Mother!... - Score: 0.6317
  11. We Need to Talk About Kevin... - Score: 0.5511
  12. Blue Jasmine... - Score: 0.5511
  13. Bourne Identity, The... - Score: 0.5463
  14. Bandits... - Score: 0.5408
  15. Medicine Man... - Score: 0.5350

Combining model scores with Reddit sentiment analysis...
Reddit scores range: 0.423 to 0.581
Model scores range: 0.535 to 0.664
The Man from Nowhere: Model(0.664) + Reddit(0.570 -> 0.655) = 0.661
Children of Heaven, The

---
# Cross-Domain – Movies Based on Books

### Load the data 
Load the data recommendations from our cross-domain recommender models, which was previously saved as a json file.

In [24]:
try: 
    books_to_movies = spark.read.json("../Models/models_results/books_to_movies.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise
books_to_movies.show(n=3, truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|book_id|recommended_movies                                                                                                                                                                                                                                                                       |
+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|167316 |[[4736, 0.89003579346618], [72737, 0.889868335097347], [1743, 0.8897560918323623], [1772, 0.8893498247382082], [311

In [25]:
# Loading book_metadata_old.json
try: 
    book_metadata = spark.read.json("../Models/within_domain_data/books_metadata_old.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

In [26]:
book_metadata = book_metadata.select(
    col("item_id").cast("long"),  # cast item_id to match recs type
    col("title")
)
book_metadata.show(3)

+--------+--------------------+
| item_id|               title|
+--------+--------------------+
|16827462|The Fault in Our ...|
| 2792775|The Hunger Games ...|
| 8812783|Mockingjay (The H...|
+--------+--------------------+
only showing top 3 rows



In [27]:
# Match movie and book ids with their titles 
books_exploded = books_to_movies.withColumn("rec", explode("recommended_movies")) \
    .select("book_id", 
            col("rec.movie_id").alias("movie_item_id"), 
            col("rec.score").alias("score"))            

# Join both metadata tables and group back
book_movies = books_exploded \
    .join(movie_metadata.select("item_id", col("title").alias("movie_title")),
          books_exploded.movie_item_id == movie_metadata.item_id, "left") \
    .join(book_metadata.select("item_id", col("title").alias("book_title")),
          books_exploded.book_id == book_metadata.item_id, "left") \
    .select("book_title", "movie_title", "score") \
    .groupBy("book_title") \
    .agg(collect_list(struct("movie_title", "score")).alias("recommended_movies_with_titles"))

book_movies.show(3, truncate=False)

+---------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|book_title                 |recommended_movies_with_titles                                                                                                                                                                                                                                                                                                                                                                                                                     |
+---------------------------+---------------------------------------

---
## Applying sentiment analysis to movies recommended based on books

In [28]:
# Show for 3 books with 10 recommended movies each. The same can be done for the other books as well.
books = book_movies.limit(3)  
books_data = books.collect()

for row in books_data:
    print(f"\nBook Title: {row['book_title']}")
    print("Top 10 Movie Recommendations:")
    for title_score in row['recommended_movies_with_titles'][:10]:
        print(f"  - {title_score['movie_title']}: {title_score['score']:.4f}")
        
print()
print(f"=== REDDIT SENTIMENT ANALYSIS ===")
print(f"Processing {len(books_data)} books for demonstration...")


Book Title: A Room of One's Own
Top 10 Movie Recommendations:
  - Summer Catch: 0.8493
  - Arguing the World: 0.8489
  - Princess and the Frog, The: 0.8488
  - Blues Brothers 2000: 0.8485
  - Stander: 0.8478
  - She-Devil: 0.8468
  - Book of Life, The: 0.8466
  - Sleepwalkers: 0.8463
  - Horseman on the Roof, The (Hussard sur le toit, Le): 0.8462
  - 'night Mother: 0.8462

Book Title: Adaptation (Adaptation, #1)
Top 10 Movie Recommendations:
  - Summer Catch: 0.8942
  - Arguing the World: 0.8938
  - Blues Brothers 2000: 0.8936
  - Stander: 0.8928
  - Princess and the Frog, The: 0.8923
  - She-Devil: 0.8919
  - 'night Mother: 0.8914
  - Sleepwalkers: 0.8912
  - Book of Life, The: 0.8909
  - Horseman on the Roof, The (Hussard sur le toit, Le): 0.8906

Book Title: Before I Go to Sleep
Top 10 Movie Recommendations:
  - Summer Catch: 0.9083
  - Blues Brothers 2000: 0.9081
  - Arguing the World: 0.9080
  - Stander: 0.9069
  - She-Devil: 0.9061
  - 'night Mother: 0.9058
  - Horseman on the R

### Collect Reddit Data

In [29]:
all_movie_titles_books = []
for book_row in books_data:
    for rec in book_row['recommended_movies_with_titles']:
        if rec['movie_title'] is not None and rec['movie_title'] not in all_movie_titles_books:
            all_movie_titles_books.append(rec['movie_title'])

# Collect Reddit data once for all movies
reddit_data_book = collect_reddit_movie_data(
    reddit,
    list(all_movie_titles_books)
)


--- Processing movie 1/11: 'Summer Catch' ---
Found 100 posts for 'Summer Catch' in r/movies
Found 18 posts for 'Summer Catch' in r/TrueFilm
Found 26 posts for 'Summer Catch' in r/moviesuggestions
Total posts found in the search: 144
The number of comments collected for 'Summer Catch' is 6
Collection complete: 144 posts, 6 comments (total: 150)

--- Processing movie 2/11: 'Arguing the World' ---
Found 100 posts for 'Arguing the World' in r/movies
Reached maximum limit of 150 posts. Stopping search.
Found 50 posts for 'Arguing the World' in r/TrueFilm
Reached maximum limit of 150 posts. Stopping search.
Total posts found in the search: 150
The number of comments collected for 'Arguing the World' is 0
Collection complete: 150 posts, 0 comments (total: 150)

--- Processing movie 3/11: 'Princess and the Frog, The' ---
Found 100 posts for 'Princess and the Frog, The' in r/movies
Found 0 posts for 'The Princess and the Frog' in r/movies
Found 10 posts for 'Princess and the Frog, The' in r/T

In [30]:
# Compute Reddit scores (for sentiment analysis) using the data collected in reddit_data_book:
reddit_scores_cache_book = calculate_reddit_scores(
    reddit_data_book,
    spark
)


Processing Summer Catch...
  Polarity: 0.152 -> Normalized: 0.576
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.576

Processing Arguing the World...
  Polarity: 0.128 -> Normalized: 0.564
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.564

Processing Princess and the Frog, The...
  Polarity: 0.148 -> Normalized: 0.574
  Volume: 140 texts -> Multiplier: 0.900
  Final Reddit score: 0.517

Processing Blues Brothers 2000...
  Polarity: 0.103 -> Normalized: 0.552
  Volume: 186 texts -> Multiplier: 1.000
  Final Reddit score: 0.552

Processing Stander...
  Polarity: 0.110 -> Normalized: 0.555
  Volume: 33 texts -> Multiplier: 0.700
  Final Reddit score: 0.388

Processing She-Devil...
  Polarity: 0.093 -> Normalized: 0.546
  Volume: 159 texts -> Multiplier: 1.000
  Final Reddit score: 0.546

Processing Book of Life, The...
  Polarity: 0.105 -> Normalized: 0.553
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.553

Processing Sleepwalkers..

In [31]:
# Process the user using the computed reddit_scores_cache and our combine_model_and_reddit_scores function:
def process_book_with_sentiment(book_title, recommended_movies, reddit_scores_cache_book):
    print(f"\n--- Processing Book: {book_title} ---")
    
    # Extract movie titles from recommendations
    movie_titles = []
    model_recommendations = []
    
    for rec in recommended_movies:
        if rec['movie_title'] is not None:
            movie_titles.append(rec['movie_title'])
            model_recommendations.append((rec['movie_title'], rec['score']))
    
    print(f"Found {len(movie_titles)} movies to analyze")
    print("Original 10 recommendations:")
    for i, (title, score) in enumerate(model_recommendations[:15]):
        print(f"  {i+1}. {title[:60]}... - Score: {score:.4f}")
    
    print("\nCombining model scores with Reddit sentiment analysis...")
    
    try:
        combined_scores = combine_model_and_reddit_scores(
            model_recommendations,
            reddit_scores_cache_book,
            model_weight=0.6,
            reddit_weight=0.4,
            no_data_penalty=0.05
        )
        return combined_scores
    except Exception as e:
        print(f"Error combining recommendations: {e}")
        return model_recommendations

# Final Recommendations - Cross Domain

In [32]:
# Process for each book and print results using process_book_with_sentiment:
print("\n" + "="*80)
print("PROCESSING EXAMPLE BOOKS")
print("="*80)
processed_books = []

for i, book_row in enumerate(books_data):
    book_title = book_row['book_title']
    recommended_movies = book_row['recommended_movies_with_titles']
    
    try:
        sentiment_enhanced_recs = process_book_with_sentiment(
            book_title,
            recommended_movies,
            reddit_scores_cache_book
        )
        
        # Store results:
        processed_books.append({
            'book_title': book_title,
            'original_recommendations': [(rec['movie_title'], rec['score']) for rec in recommended_movies if rec['movie_title'] is not None],
            'sentiment_enhanced_recommendations': sentiment_enhanced_recs
        })
        
        # Show comparison of previous scores with the final ones:
        print(f"\n{'='*60}")
        print(f"RESULTS FOR BOOK: {book_title}")
        print(f"{'='*60}")
        
        print("\nTOP 10 COMPARISON:")
        print(f"{'Rank':<4} | {'Original Score':<13} | {'Enhanced Score':<13} | {'Change':<8} | {'Title'}")
        print("-" * 100)
        
        original_recs = processed_books[-1]['original_recommendations'][:10]
        enhanced_recs = sentiment_enhanced_recs[:10]
        
        for j in range(builtins.min(10, len(original_recs), len(enhanced_recs))):
            orig_score = original_recs[j][1]
            enh_score = enhanced_recs[j][1]
            change = enh_score - orig_score
            change_str = f"{change:+.4f}"
            title = enhanced_recs[j][0][:50] + "..." if len(enhanced_recs[j][0]) > 50 else enhanced_recs[j][0]
            
            print(f"{j+1:<4} | {orig_score:<13.4f} | {enh_score:<13.4f} | {change_str:<8} | {title}")
        
    except Exception as e:
        print(f"Error processing book {book_title}: {e}")
        continue
        

# Create PySpark DataFrame from results:

# Prepare data for PySpark DataFrame
spark_data = []
for book_data in processed_books:
    book_title = book_data['book_title']
    
    # Flatten original recommendations
    original_titles = [rec[0] for rec in book_data['original_recommendations']]
    original_scores = [rec[1] for rec in book_data['original_recommendations']]
    
    # Flatten enhanced recommendations
    enhanced_titles = [rec[0] for rec in book_data['sentiment_enhanced_recommendations']]
    enhanced_scores = [rec[1] for rec in book_data['sentiment_enhanced_recommendations']]
    
    spark_data.append({
        'book_title': book_title,
        'original_titles': original_titles,
        'original_scores': original_scores,
        'enhanced_titles': enhanced_titles,
        'enhanced_scores': enhanced_scores,
        'top_10_recommendations': enhanced_titles[:10]
    })

schema = StructType([
    StructField("book_title", StringType(), True),
    StructField("original_titles", ArrayType(StringType()), True),
    StructField("original_scores", ArrayType(FloatType()), True),
    StructField("enhanced_titles", ArrayType(StringType()), True),
    StructField("enhanced_scores", ArrayType(FloatType()), True),
    StructField("top_10_recommendations", ArrayType(StringType()), True)
])

# Create PySpark DataFrame
cross_recommendations_df_movies = spark.createDataFrame(spark_data, schema)
cross_recommendations_df_movies.show(truncate=False)

print()
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS ORDER")
print("="*80)

# Collect results 
results = cross_recommendations_df_movies.collect()
for row in results:
    book_title = row['book_title']
    top_10 = row['top_10_recommendations']
    
    print(f"\nTOP 10 MOVIE RECOMMENDATIONS FOR BOOK: {book_title}")
    print("-" * 50)
    for i, title in enumerate(top_10, 1):
        print(f"{i:2d}. {title}")


PROCESSING EXAMPLE BOOKS

--- Processing Book: A Room of One's Own ---
Found 10 movies to analyze
Original 10 recommendations:
  1. Summer Catch... - Score: 0.8493
  2. Arguing the World... - Score: 0.8489
  3. Princess and the Frog, The... - Score: 0.8488
  4. Blues Brothers 2000... - Score: 0.8485
  5. Stander... - Score: 0.8478
  6. She-Devil... - Score: 0.8468
  7. Book of Life, The... - Score: 0.8466
  8. Sleepwalkers... - Score: 0.8463
  9. Horseman on the Roof, The (Hussard sur le toit, Le)... - Score: 0.8462
  10. 'night Mother... - Score: 0.8462

Combining model scores with Reddit sentiment analysis...
Reddit scores range: 0.388 to 0.576
Model scores range: 0.846 to 0.849
Summer Catch: Model(0.849) + Reddit(0.576 -> 0.849) = 0.849
Arguing the World: Model(0.849) + Reddit(0.564 -> 0.849) = 0.849
Princess and the Frog, The: Model(0.849) + Reddit(0.517 -> 0.848) = 0.849
Blues Brothers 2000: Model(0.849) + Reddit(0.552 -> 0.849) = 0.849
Stander: Model(0.848) + Reddit(0.388 -> 0.8